In [ ]:
%pip install scikit-learn

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models,regularizers
from sklearn.model_selection import train_test_split


In [ ]:

# ============================================================
# 1. Preprocessing (Letterbox Padding for varying custom width/height)
# ============================================================

def tf_preprocess_wrapper(image_path, d1, d2, d3, d4, d5):
    [img,] = tf.py_function(lambda p: preprocess_image_np(p), [image_path], [tf.float32])
    img.set_shape((40, 160, 1))
    
    # Bundle labels back together for multi-output training
    labels = {"digit1": d1, "digit2": d2, "digit3": d3, "digit4": d4, "digit5": d5}
    return img, labels


In [ ]:

# ============================================================
# 2. Building Multi-Output Data Pipeline
# ============================================================
def preprocess_image_np(image_path, target_w=160, target_h=40):
    """Loads image, converts to grayscale, and pads preserving aspect ratio."""
    # --- CRITICAL FIX 1: Convert TensorFlow bytes back to a standard Python string ---
    if isinstance(image_path, bytes):
        image_path = image_path.decode('utf-8')
    elif hasattr(image_path, 'numpy'):  # Extra fallback if it's a eager tensor object
        image_path = image_path.numpy().decode('utf-8')
        
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        # --- SAFETY CHECK 1: File couldn't be loaded or is corrupted ---
        if img is None or img.size == 0:
            print(f"⚠️ Warning: Unreadable image skipped -> {image_path}")
            return np.zeros((target_h, target_w, 1), dtype=np.float32)
        
        h, w = img.shape[:2]
        
        # --- SAFETY CHECK 2: Zero-dimension handling ---
        if h == 0 or w == 0:
            print(f"⚠️ Warning: Invalid image dimensions -> {image_path} ({w}x{h})")
            return np.zeros((target_h, target_w, 1), dtype=np.float32)
            
        scale = min(target_w / w, target_h / h)
        nw, nh = int(w * scale), int(h * scale)
        
        if nw <= 0 or nh <= 0:
            return np.zeros((target_h, target_w, 1), dtype=np.float32)
            
        resized = cv2.resize(img, (nw, nh))
        
        canvas = np.full((target_h, target_w), 128, dtype=np.uint8)
        x_offset = (target_w - nw) // 2
        y_offset = (target_h - nh) // 2
        canvas[y_offset:y_offset+nh, x_offset:x_offset+nw] = resized
        
        img_tensor = canvas.astype(np.float32) / 255.0
        img_tensor = np.expand_dims(img_tensor, axis=-1) 
        return img_tensor

    except Exception as e:
        print(f"💥 Error processing file {image_path}: {str(e)}")
        return np.zeros((target_h, target_w, 1), dtype=np.float32)
def create_tf_dataset(df, image_folder, batch_size=32, is_training=True):
    file_paths = [os.path.join(image_folder, fname) for fname in df['image']]
    
    # Ensure strings are padded properly (e.g. "5" -> "00005")
    df['label'] = df['label'].astype(str).str.zfill(5)
    
    # Split the string labels into 5 distinct arrays of digits
    d1 = df['label'].str[0].astype(int).values
    d2 = df['label'].str[1].astype(int).values
    d3 = df['label'].str[2].astype(int).values
    d4 = df['label'].str[3].astype(int).values
    d5 = df['label'].str[4].astype(int).values
    
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, d1, d2, d3, d4, d5))
    dataset = dataset.map(tf_preprocess_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(df)).repeat()
        
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:


# ============================================================
# 3. Keras Custom 5-Digit Sequence Model
# ============================================================

def build_multi_digit_cnn():
    inputs = layers.Input(shape=(40, 160, 1), name="input_layer")
    
    # Shared Feature Extraction Base
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # Output: 20 x 80 x 32
    
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # Output: 10 x 40 x 64
    
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Important: Horizontal structure-preserving pooling
    shared_features = layers.MaxPooling2D(pool_size=(2, 1), strides=(2, 1))(x) # Output: 5 x 40 x 128
    
    # Global contextual representation (like your ResNet notebook workflow)
    global_pooled = layers.GlobalAveragePooling2D(name="global_pool")(shared_features)
    
    # Helper to construct identical separate digit heads
    def build_digit_head(name):
        h = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(global_pooled)
        h = layers.BatchNormalization()(h)
        h = layers.Dropout(0.3)(h)
        return layers.Dense(10, activation="softmax", name=name)(h)
        
    # Generate 5 unique branched outputs
    digit1 = build_digit_head("digit1")
    digit2 = build_digit_head("digit2")
    digit3 = build_digit_head("digit3")
    digit4 = build_digit_head("digit4")
    digit5 = build_digit_head("digit5")
    
    # Define functional model with multiple heads
    model = tf.keras.Model(inputs=inputs, outputs=[digit1, digit2, digit3, digit4, digit5])
    return model


In [ ]:

# ============================================================
# 4. Main Training Script Setup
# ============================================================
if __name__ == "__main__":
    CSV_FILE = "../data_set_generator_for_cnns_only/data_set/train.csv"
    IMAGE_FOLDER = "..data_set_genrator_for_cnns_only/data_set/train"
    df = pd.read_csv(CSV_FILE)
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    
    BATCH_SIZE = 32
    train_ds = create_tf_dataset(train_df, IMAGE_FOLDER, batch_size=BATCH_SIZE, is_training=True)
    val_ds = create_tf_dataset(val_df, IMAGE_FOLDER, batch_size=BATCH_SIZE, is_training=False)
    
    model = build_multi_digit_cnn()
    
    # Loss mapping matches the specific key names assigned to output layers
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss={
            "digit1": loss_fn,
            "digit2": loss_fn,
            "digit3": loss_fn,
            "digit4": loss_fn,
            "digit5": loss_fn,
        },
        metrics={
        "digit1": "accuracy",
        "digit2": "accuracy",
        "digit3": "accuracy",
        "digit4": "accuracy",
        "digit5": "accuracy",
    }
    )
    
    model.summary()
    
    steps_per_epoch = len(train_df) // BATCH_SIZE
    
   
    


In [ ]:
model.fit(
        train_ds,
        epochs=30,
        steps_per_epoch=steps_per_epoch if steps_per_epoch > 0 else 1,
        validation_data=val_ds
    )

In [ ]:
model.save("custom_5digit_meter_model.keras")